[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/084.%20The%20Multi-Facility%20Location%20-%20p-Center%20Problem/P84-Tier-9_executed.ipynb)

---


# 84. The Multi-Facility Location: p-Center Problem

## Tier 9: QAOA and the Limits of Statevector Simulation

### Purpose and scope

This notebook formulates the discrete p-center problem as a quadratic unconstrained binary optimization (QUBO) model and explains how the Quantum Approximate Optimization Algorithm (QAOA) uses that model. It also includes an exact statevector simulator for small QUBOs and an exact enumeration benchmark for the small facility-location instance used here.

The supplied instance has 8 candidate facilities, 6 demand points, and requires selecting 3 facilities. The QUBO also includes assignment and radius-selection variables, so its number of binary variables is much larger than the number of facility-selection decisions. A statevector stores an amplitude for every binary string; consequently, the exact simulator checks the required memory before attempting to allocate a state. For this instance, it reports the resource requirement rather than fabricating a QAOA result.

### Model assumptions

- Demand points and candidate facility locations are fixed points in a two-dimensional Euclidean plane.
- Exactly `p` candidate locations are opened, and each demand point is assigned to one opened facility.
- The objective is to minimize the largest assigned Euclidean distance.
- A demand point may be assigned to any open facility. At an optimum, assignments can be chosen so that each demand point is served by a nearest open facility.
- The QUBO uses the distinct demand-to-facility distances as possible radius values. It does not model capacities, opening costs, road networks, uncertainty, or facility construction decisions beyond selecting locations.

### Method overview

The solution sequence is: construct the distance matrix; build an exact QUBO for the p-center objective and constraints; calculate an exact small-instance benchmark by enumerating facility subsets; then assess whether exact statevector QAOA simulation is feasible for the QUBO size. QAOA alternates a problem-dependent phase operation and a mixing operation, with their angles optimized by a classical optimizer. The implementation is a statevector simulator, not a quantum-hardware execution.


In [ ]:
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, List, Optional, Set, Tuple
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
plt.style.use("default")
sns.set_palette("husl")

## 1. Problem data and the p-center objective

Let `I` be the set of demand points and `J` the set of candidate facilities. The distance from demand point `i` to candidate facility `j` is `d[i, j]`. A selected facility set `S` must have exactly `p` members. The p-center objective is

$$
\min_{S\subseteq J,\,|S|=p}\;\max_{i\in I}\;\min_{j\in S} d_{ij}.
$$

For each demand point, the inner minimum is its distance to the nearest selected facility. The outer maximum is the distance of the least well-served demand point. Minimizing this maximum limits the worst-case service distance. `compute_distance_matrix` implements the Euclidean distance, and `evaluate_solution` evaluates this objective for a proposed facility set.


In [ ]:
@dataclass
class PCenterInstance:
    """A p-center instance with point coordinates and a facility count."""

    demand_points: List[Tuple[float, float]]
    facility_locations: List[Tuple[float, float]]
    p: int

    def __post_init__(self) -> None:
        if not 1 <= self.p <= len(self.facility_locations):
            raise ValueError("p must be between 1 and the number of candidate facilities.")
        if not self.demand_points:
            raise ValueError("At least one demand point is required.")
        if not self.facility_locations:
            raise ValueError("At least one candidate facility is required.")

    def compute_distance_matrix(self) -> np.ndarray:
        """Return Euclidean distances with rows for demand and columns for facilities."""
        demand = np.asarray(self.demand_points, dtype=float)
        facilities = np.asarray(self.facility_locations, dtype=float)
        return np.linalg.norm(demand[:, None, :] - facilities[None, :, :], axis=2)

    def evaluate_solution(self, facility_set: Set[int]) -> float:
        """Return the maximum nearest-open-facility distance for a valid facility set."""
        if len(facility_set) != self.p:
            raise ValueError("A solution must select exactly p facilities.")
        if any(j < 0 or j >= len(self.facility_locations) for j in facility_set):
            raise ValueError("Facility indices must refer to candidate locations.")
        distances = self.compute_distance_matrix()
        return float(np.max(np.min(distances[:, sorted(facility_set)], axis=1)))

    def get_optimal_solution(self) -> Tuple[Set[int], float]:
        """Find an exact optimum by enumerating all p-facility subsets."""
        best_solution: Optional[Set[int]] = None
        best_objective = float("inf")
        for subset in combinations(range(len(self.facility_locations)), self.p):
            facility_set = set(subset)
            objective = self.evaluate_solution(facility_set)
            if objective < best_objective:
                best_solution = facility_set
                best_objective = objective
        assert best_solution is not None
        return best_solution, best_objective

## 2. Exact QUBO formulation

A QUBO represents a problem using binary variables and a quadratic energy function. Binary variables take values in `{0, 1}`. For this p-center model, use:

- `x[j] = 1` if candidate facility `j` is selected;
- `y[i, j] = 1` if demand point `i` is assigned to facility `j`;
- `z[k] = 1` if radius candidate `r[k]` is selected.

The radius candidates are the distinct values in the distance matrix. A feasible selection of facilities and assignments determines a radius at least as large as every assigned distance. Conversely, if the chosen radius is at least every assigned distance, it bounds the maximum service distance. Minimizing the selected radius therefore represents the p-center objective exactly.

The penalty terms enforce the constraints:

$$
\begin{aligned}
E(x,y,z) ={}& \sum_k r_k z_k
+ A\left(\sum_j x_j-p\right)^2
+ B\sum_i\left(\sum_j y_{ij}-1\right)^2\\
+{}& B\sum_{i,j} y_{ij}(1-x_j)
+ B\sum_{i,j}\sum_{k:\,r_k<d_{ij}} y_{ij}z_k
+ B\left(\sum_k z_k-1\right)^2.
\end{aligned}
$$

The first penalty enforces exactly `p` selected facilities; the second enforces exactly one assigned facility per demand point; the third prevents assignment to a closed facility; and the fourth prevents an assignment whose distance exceeds the selected radius. The final penalty selects exactly one radius. Each expression is quadratic after expansion because the variables are binary and satisfy `v² = v`.

The penalty coefficient is chosen as `B = max_distance + 1`. The feasible objective lies between zero and `max_distance`; any violated constraint incurs a penalty greater than that entire objective range. Thus an infeasible assignment cannot be preferred to a feasible one on account of the radius objective. The QUBO matrix uses the convention `E = constant + v.T @ Q @ v`; off-diagonal coefficients are split equally between `Q[a, b]` and `Q[b, a]`.


In [ ]:
@dataclass
class QUBOModel:
    """Quadratic binary energy E(v) = constant + v.T @ Q @ v."""

    Q: np.ndarray
    constant: float
    variable_names: List[str]
    n_facilities: int
    n_demand: int
    radius_values: np.ndarray

    def energy(self, bits: np.ndarray) -> float:
        bits = np.asarray(bits, dtype=float)
        return float(self.constant + bits @ self.Q @ bits)


def formulate_qubo(instance: PCenterInstance) -> QUBOModel:
    """Build the exact radius-and-assignment QUBO for a p-center instance."""
    distances = instance.compute_distance_matrix()
    n_demand, n_facilities = distances.shape
    radius_values = np.unique(distances)
    n_radii = len(radius_values)
    total_vars = n_facilities + n_demand * n_facilities + n_radii
    Q = np.zeros((total_vars, total_vars), dtype=float)
    constant = 0.0
    penalty = float(np.max(distances) + 1.0)

    x_index = lambda j: j
    y_index = lambda i, j: n_facilities + i * n_facilities + j
    z_index = lambda k: n_facilities + n_demand * n_facilities + k

    def add_linear(index: int, coefficient: float) -> None:
        Q[index, index] += coefficient

    def add_pair(first: int, second: int, coefficient: float) -> None:
        if first == second:
            Q[first, first] += coefficient
        else:
            Q[first, second] += coefficient / 2.0
            Q[second, first] += coefficient / 2.0

    def add_square(coefficients: Dict[int, float], offset: float, weight: float) -> None:
        nonlocal constant
        constant += weight * offset**2
        for index, coefficient in coefficients.items():
            add_linear(index, weight * (coefficient**2 + 2.0 * offset * coefficient))
        indices = list(coefficients)
        for position, first in enumerate(indices):
            for second in indices[position + 1:]:
                add_pair(first, second, 2.0 * weight * coefficients[first] * coefficients[second])

    variable_names = [f"x[{j}]" for j in range(n_facilities)]
    variable_names.extend(
        f"y[{i},{j}]" for i in range(n_demand) for j in range(n_facilities)
    )
    variable_names.extend(f"z[{k}]" for k in range(n_radii))

    # The squared cardinality residual is zero precisely when p facilities are selected.
    add_square({x_index(j): 1.0 for j in range(n_facilities)}, -instance.p, penalty)

    for i in range(n_demand):
        # One-hot assignment ensures every demand point has exactly one serving facility.
        add_square({y_index(i, j): 1.0 for j in range(n_facilities)}, -1.0, penalty)
        for j in range(n_facilities):
            # This nonnegative quadratic penalty is active only when y[i,j]=1 and x[j]=0.
            add_linear(y_index(i, j), penalty)
            add_pair(y_index(i, j), x_index(j), -penalty)
            # A selected radius must cover every distance used by an assignment.
            for k, radius in enumerate(radius_values):
                if radius < distances[i, j]:
                    add_pair(y_index(i, j), z_index(k), penalty)

    # One-hot radius selection and the radius objective.
    add_square({z_index(k): 1.0 for k in range(n_radii)}, -1.0, penalty)
    for k, radius in enumerate(radius_values):
        add_linear(z_index(k), float(radius))

    return QUBOModel(
        Q=Q,
        constant=constant,
        variable_names=variable_names,
        n_facilities=n_facilities,
        n_demand=n_demand,
        radius_values=radius_values,
    )

## 3. QAOA statevector simulation

A QAOA state is a complex amplitude vector over all `2**n` bitstrings for an `n`-variable QUBO. Each layer applies a phase based on the QUBO energy, followed by an X-mixer rotation on every qubit. In mathematical notation, one layer applies a problem unitary `exp(-i γ H_C)` and a mixer unitary `exp(-i β H_M)`, where `H_C` is diagonal with the QUBO energies and `H_M = Σ_q X_q`. The parameters `γ` and `β` are optimized classically to minimize the expected QUBO energy.

This exact statevector representation has exponential memory use: storing `2**n` complex amplitudes requires approximately `16 × 2**n` bytes when each complex number uses 16 bytes, before temporary arrays and other overhead. The simulator below therefore has a configurable qubit limit. If the QUBO exceeds that limit, it raises a clear exception before allocating the state. This is a resource check, not a claim that a quantum device has been run.


In [ ]:
class QuantumQAOASolver:
    """Exact statevector QAOA simulator for QUBOs small enough to fit in memory."""

    def __init__(self, model: QUBOModel, qaoa_layers: int = 2, max_qubits: int = 20):
        if qaoa_layers < 1:
            raise ValueError("qaoa_layers must be at least 1.")
        self.model = model
        self.qaoa_layers = qaoa_layers
        self.max_qubits = max_qubits
        self.n_qubits = len(model.variable_names)
        self.parameter_history: List[np.ndarray] = []
        self.objective_history: List[float] = []

    def _check_statevector_size(self) -> None:
        if self.n_qubits > self.max_qubits:
            estimated_gib = 16 * (2**self.n_qubits) / (1024**3)
            raise MemoryError(
                f"Exact statevector simulation requires {self.n_qubits} qubits "
                f"(about {estimated_gib:.2g} GiB for amplitudes alone); "
                f"the configured limit is {self.max_qubits}. No state was allocated."
            )

    def quantum_state_evolution(self, params: np.ndarray) -> np.ndarray:
        """Apply alternating QUBO phase and standard X-mixer unitaries."""
        self._check_statevector_size()
        params = np.asarray(params, dtype=float)
        if params.size != 2 * self.qaoa_layers:
            raise ValueError("The parameter vector must contain gamma and beta per layer.")

        dimension = 2**self.n_qubits
        state = np.full(dimension, 1.0 / np.sqrt(dimension), dtype=complex)
        bit_indices = np.arange(dimension, dtype=np.uint64)
        energies = np.empty(dimension, dtype=float)
        for basis_index in range(dimension):
            bits = ((bit_indices[basis_index] >> np.arange(self.n_qubits)) & 1).astype(float)
            energies[basis_index] = self.model.energy(bits)

        for layer in range(self.qaoa_layers):
            gamma = params[2 * layer]
            beta = params[2 * layer + 1]
            state *= np.exp(-1j * gamma * energies)

            # Applying each single-qubit X rotation preserves the unitary mixer exactly.
            cosine = np.cos(beta)
            sine = -1j * np.sin(beta)
            for qubit in range(self.n_qubits):
                paired = state.reshape((2,) * self.n_qubits)
                paired = np.moveaxis(paired, qubit, 0)
                first = paired[0].copy()
                second = paired[1].copy()
                paired[0] = cosine * first + sine * second
                paired[1] = sine * first + cosine * second
                state = np.moveaxis(paired, 0, qubit).reshape(-1)

        return state

    def expected_energy(self, params: np.ndarray) -> float:
        """Return the exact expected QUBO energy for a parameter vector."""
        state = self.quantum_state_evolution(params)
        probabilities = np.abs(state) ** 2
        dimension = len(state)
        energies = np.empty(dimension, dtype=float)
        for basis_index in range(dimension):
            bits = ((basis_index >> np.arange(self.n_qubits)) & 1).astype(float)
            energies[basis_index] = self.model.energy(bits)
        return float(np.dot(probabilities, energies))

    def solve(self, seed: int = 42, maxiter: int = 50) -> Dict[str, object]:
        """Optimize QAOA angles and return the lowest-energy measured bitstring."""
        self._check_statevector_size()
        rng = np.random.default_rng(seed)
        initial_params = rng.uniform(0.0, np.pi, 2 * self.qaoa_layers)

        def callback(params: np.ndarray) -> None:
            self.parameter_history.append(params.copy())
            self.objective_history.append(self.expected_energy(params))

        result = minimize(
            self.expected_energy,
            initial_params,
            method="Nelder-Mead",
            callback=callback,
            options={"maxiter": maxiter, "disp": False},
        )
        state = self.quantum_state_evolution(result.x)
        probabilities = np.abs(state) ** 2
        bit_index = int(np.argmax(probabilities))
        bits = ((bit_index >> np.arange(self.n_qubits)) & 1).astype(int)
        return {
            "bits": bits,
            "energy": self.model.energy(bits),
            "expected_energy": float(result.fun),
            "iterations": int(result.nit),
            "parameters": result.x,
            "converged": bool(result.success),
        }

## 4. Construct the instance and calculate an exact benchmark

The fixed random seed makes the coordinate generation reproducible. Exact subset enumeration is practical for this small instance and provides a valid benchmark for interpreting any approximate method. It is not a branch-and-bound computation, and its timing should not be interpreted as a comparison with QAOA: the two procedures perform different work and the statevector simulation is not run when its memory requirement exceeds the configured limit.


In [ ]:
np.random.seed(42)
n_facilities = 8
n_demand = 6
p = 3

facility_locations = [
    (float(np.random.uniform(0, 10)), float(np.random.uniform(0, 10)))
    for _ in range(n_facilities)
]
demand_points = [
    (float(np.random.uniform(0, 10)), float(np.random.uniform(0, 10)))
    for _ in range(n_demand)
]
instance = PCenterInstance(demand_points, facility_locations, p)
qubo_model = formulate_qubo(instance)

print("p-Center Instance")
print("=" * 24)
print(f"Candidate facilities: {n_facilities}")
print(f"Demand points: {n_demand}")
print(f"Facilities to select: {p}")
print(f"Facility subsets in exact benchmark: {math.comb(n_facilities, p):,}")
print(f"QUBO binary variables: {len(qubo_model.variable_names)}")
print(f"QAOA statevector amplitudes required: 2^{len(qubo_model.variable_names)}")

start_time = time.perf_counter()
classical_solution, classical_objective = instance.get_optimal_solution()
classical_time = time.perf_counter() - start_time

print("\nExact subset-enumeration benchmark")
print(f"Selected facility indices: {sorted(classical_solution)}")
print(f"Optimal maximum service distance: {classical_objective:.6f} distance units")
print(f"Enumeration time: {classical_time:.6f} seconds")

qaoa_solver = QuantumQAOASolver(qubo_model, qaoa_layers=2, max_qubits=20)
try:
    qaoa_results = qaoa_solver.solve(seed=42, maxiter=50)
    print("\nQAOA statevector result")
    print(f"Lowest-energy measured QUBO value: {qaoa_results['energy']:.6f}")
    print(f"Optimizer iterations: {qaoa_results['iterations']}")
    print(f"Optimizer reported convergence: {qaoa_results['converged']}")
except MemoryError as error:
    qaoa_results = None
    print("\nQAOA statevector simulation not run")
    print(error)

assert len(classical_solution) == instance.p
assert math.isclose(
    instance.evaluate_solution(classical_solution), classical_objective, rel_tol=1e-12, abs_tol=1e-12
)

## 5. Interpret the benchmark and the QAOA resource check

The exact benchmark is the minimum over all subsets containing exactly `p` facilities. It gives the optimal objective for the specified finite instance. It does not establish a quantum speedup. The QUBO model has additional binary variables for assignments and radius selection, and exact statevector simulation grows exponentially with that total variable count—not merely with the number of candidate facilities.

If the configured statevector limit is exceeded, `qaoa_results` is `None` and no QAOA objective, runtime, optimality gap, measurement distribution, or speedup is reported. If a QAOA run is feasible for a smaller model, its result must be decoded and checked against the original p-center constraints before being interpreted as a p-center solution. A low QUBO energy alone is not a substitute for that feasibility check.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

demand_x, demand_y = zip(*demand_points)
facility_x, facility_y = zip(*facility_locations)
selected_x = [facility_locations[j][0] for j in sorted(classical_solution)]
selected_y = [facility_locations[j][1] for j in sorted(classical_solution)]

ax.scatter(demand_x, demand_y, marker="s", s=90, color="tab:red", label="Demand points")
ax.scatter(facility_x, facility_y, marker="o", s=65, color="tab:gray", label="Candidate facilities")
ax.scatter(selected_x, selected_y, marker="^", s=150, color="tab:green", label="Selected facilities")

distances = instance.compute_distance_matrix()
for i, demand in enumerate(demand_points):
    nearest = min(classical_solution, key=lambda j: distances[i, j])
    facility = facility_locations[nearest]
    ax.plot(
        [demand[0], facility[0]],
        [demand[1], facility[1]],
        linestyle="--",
        linewidth=1,
        color="black",
        alpha=0.4,
    )
    ax.annotate(f"D{i + 1}", demand, xytext=(4, 4), textcoords="offset points", fontsize=8)

for j, location in enumerate(facility_locations):
    ax.annotate(f"F{j + 1}", location, xytext=(4, -10), textcoords="offset points", fontsize=8)

ax.set_title("Exact p-Center Benchmark: Selected Facilities and Assignments")
ax.set_xlabel("X coordinate (distance units)")
ax.set_ylabel("Y coordinate (distance units)")
ax.axis("equal")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Qubit counts and computational interpretation

For `n` QUBO variables, an exact statevector has `2**n` complex amplitudes. At 16 bytes per complex amplitude, the amplitudes alone require approximately `16 × 2**n` bytes. This is a memory estimate for a classical statevector representation; it is not the amount of memory used by quantum hardware, which does not store a classical array of all amplitudes.

The facility subset search space has size `choose(|J|, p)`, while the QUBO statevector dimension is `2**n`, where `n` includes facility, assignment, and radius variables. These quantities describe different representations and do not imply that a quantum algorithm evaluates all states and returns the best one. Superposition, interference, and measurement are components of quantum algorithms, but none by itself guarantees a speedup or an optimal measurement outcome. QAOA performance depends on the encoding, circuit, parameter optimization, measurement budget, and hardware or simulator implementation.


In [ ]:
def statevector_memory_gib(n_qubits: int) -> float:
    """Estimate memory for complex128 state amplitudes, excluding overhead."""
    return 16 * (2**n_qubits) / (1024**3)


problem_sizes = [
    (4, 3, 2),
    (6, 4, 3),
    (8, 6, 3),
]

print("Facility-subset and statevector size estimates")
print("=" * 67)
print(f"{'Facilities':>10} {'Demand':>8} {'p':>4} {'Subsets':>12} {'QUBO bits':>11} {'Amplitude GiB':>15}")
print("-" * 67)
for facility_count, demand_count, facility_count_to_open in problem_sizes:
    np.random.seed(42)
    facilities = [
        (float(np.random.uniform(0, 10)), float(np.random.uniform(0, 10)))
        for _ in range(facility_count)
    ]
    demands = [
        (float(np.random.uniform(0, 10)), float(np.random.uniform(0, 10)))
        for _ in range(demand_count)
    ]
    sample_instance = PCenterInstance(demands, facilities, facility_count_to_open)
    sample_model = formulate_qubo(sample_instance)
    qubits = len(sample_model.variable_names)
    subset_count = math.comb(facility_count, facility_count_to_open)
    print(
        f"{facility_count:>10} {demand_count:>8} {facility_count_to_open:>4} "
        f"{subset_count:>12,} {qubits:>11} {statevector_memory_gib(qubits):>15.3g}"
    )

## 7. Hardware and implementation considerations

A hardware implementation would require a circuit encoding the QUBO phase operation and a mixer, plus repeated circuit executions to estimate outcomes. Practical feasibility depends on the number of logical variables, circuit connectivity, gate errors, coherence, compilation overhead, and the number of measurement shots. Hardware specifications vary by provider and change over time; this notebook does not assume a particular device or invent device benchmarks.

The simulator above applies ideal unitary operations and exact probability calculations. It does not model hardware noise, circuit compilation, finite-shot uncertainty, error mitigation, or quantum execution time. Its `max_qubits` check limits classical memory use for statevector simulation; it is not a hardware-capability assessment.

### Key takeaways

- The p-center objective minimizes the worst nearest-facility distance.
- Facility selection, demand assignment, and radius selection produce an exact quadratic binary energy model with explicit penalties.
- QAOA uses alternating problem and mixer unitaries with classically optimized parameters.
- Exact statevector simulation is exponential in the total number of QUBO variables.
- An exact classical benchmark on a small instance does not demonstrate quantum advantage, and a statevector resource estimate is not a quantum speedup claim.

### Further study

Natural extensions include adding operational constraints to the p-center model, examining the effects of alternative penalty scales while preserving feasibility, and assessing QAOA on instances small enough for the chosen simulator or available hardware. Any reported performance comparison should use clearly specified instances, consistent timing boundaries, feasibility checks, and repeated measurements where sampling is involved.
